chunk the nhs dataset by markdown and save to a file.

In [14]:
import json
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup, NavigableString, Tag

In [15]:
import re


def chunk_by_markdown(record):
    """
    Split a markdown document into chunks using ## headings.

    Parameters
    ----------
    record : dict

    Returns
    -------
    list[dict]
    """

    text = record["content"]

    # split by H2 and H3
    parts = re.split(r"(?=^#{2,3}\s)", text, flags=re.MULTILINE)    

    parts = [p for p in parts if p.strip()]

    chunks = []

    for i, part in enumerate(parts):

        part = part.strip()

        if not part:
            continue

        heading = ""

        first_line = part.splitlines()[0]

        heading = re.sub(r"^#{2,3}\s*", "", first_line).strip() # Remove the Markdown heading symbols from the beginning of the line.  

        chunks.append({
            "chunk_id": f"{record['id']}-{i+1}",
            "parent_id": record["id"],
            "category": record["category"],
            "section": record["section"],
            "heading": heading,
            "url": record["url"],
            "content": part,
        })

    return chunks

In [16]:
import json

# Load records
with open("../data/nhs-symptom.json", "r", encoding="utf-8") as f:
    records = json.load(f)

# Chunk all records
all_chunks = []

for record in records:
    chunks = chunk_by_markdown(record)
    all_chunks.extend(chunks)

# Save chunks
with open("../data/nhs-symptom-chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"Processed {len(records)} records")
print(f"Created {len(all_chunks)} chunks")
print("Saved to ../data/nhs-symptom-chunks.json")

Processed 425 records
Created 6855 chunks
Saved to ../data/nhs-symptom-chunks.json


In [17]:
all_chunks[0]    

{'chunk_id': 'abdominal-aortic-aneurysm-1',
 'parent_id': 'abdominal-aortic-aneurysm',
 'category': 'symptom',
 'section': 'Abdominal aortic aneurysm',
 'heading': '# Abdominal aortic aneurysm',
 'url': 'https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/abdominal-aortic-aneurysm/',
 'content': '# Abdominal aortic aneurysm\n\n- About abdominal aortic aneurysms\n- Symptoms of an abdominal aortic aneurysm\n- Causes of an abdominal aortic aneurysm\n- Diagnosing an abdominal aortic aneurysm\n- Treating an abdominal aortic aneurysm\n- Preventing an abdominal aortic aneurysm'}

In [18]:
import json

headache_chunks = [
    chunk for chunk in all_chunks
    if chunk["parent_id"] == "headaches"
]

print(json.dumps(headache_chunks, indent=2, ensure_ascii=False))

[
  {
    "chunk_id": "headaches-1",
    "parent_id": "headaches",
    "category": "symptom",
    "section": "Headaches",
    "heading": "# Headaches",
    "url": "https://www.nhsinform.scot/illnesses-and-conditions/brain-nerves-and-spinal-cord/headaches/",
    "content": "# Headaches\n\nMost headaches are not serious. In many cases, you can treat your headache at home."
  },
  {
    "chunk_id": "headaches-2",
    "parent_id": "headaches",
    "category": "symptom",
    "section": "Headaches",
    "heading": "Phone 999 or go to A&E if you have a headache and:",
    "url": "https://www.nhsinform.scot/illnesses-and-conditions/brain-nerves-and-spinal-cord/headaches/",
    "content": "### Phone 999 or go to A&E if you have a headache and:\n\n- it began after a severe head injury\n- it started suddenly and is very severe – it may feel like a blinding pain\n- new sudden symptoms of a stroke , like being unable to walk, having weakness down one side of the face or body, or slurred speech\n- s

In [19]:
headache_chunks = [
    chunk for chunk in all_chunks
    if chunk["parent_id"] == "headaches"
]

for i, chunk in enumerate(headache_chunks, start=1):
    print(f"Chunk {i}")
    print(chunk["content"])
    print("=" * 80)

Chunk 1
# Headaches

Most headaches are not serious. In many cases, you can treat your headache at home.
Chunk 2
### Phone 999 or go to A&E if you have a headache and:

- it began after a severe head injury
- it started suddenly and is very severe – it may feel like a blinding pain
- new sudden symptoms of a stroke , like being unable to walk, having weakness down one side of the face or body, or slurred speech
- symptoms of meningitis or sepsis , like a high fever with a stiff neck
- are not responding normally, or are very drowsy and struggling to stay awake
- visual problems that are new and different to any you’ve experienced before, and are not related to a migraine
Chunk 3
### Contact your GP practice immediately if you have a headache and:

- also feel more confused than usual
- the whole of your eye is red
- changes in your vision
- it does not go away and gets worse over time
- is triggered suddenly by coughing, laughing, sneezing, changing posture, or physical effort
- as wel

In [20]:
import json

print(json.dumps(all_chunks[0], indent=2))


{
  "chunk_id": "abdominal-aortic-aneurysm-1",
  "parent_id": "abdominal-aortic-aneurysm",
  "category": "symptom",
  "section": "Abdominal aortic aneurysm",
  "heading": "# Abdominal aortic aneurysm",
  "url": "https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/abdominal-aortic-aneurysm/",
  "content": "# Abdominal aortic aneurysm\n\n- About abdominal aortic aneurysms\n- Symptoms of an abdominal aortic aneurysm\n- Causes of an abdominal aortic aneurysm\n- Diagnosing an abdominal aortic aneurysm\n- Treating an abdominal aortic aneurysm\n- Preventing an abdominal aortic aneurysm"
}


In [21]:
print(all_chunks[0]['content'])

# Abdominal aortic aneurysm

- About abdominal aortic aneurysms
- Symptoms of an abdominal aortic aneurysm
- Causes of an abdominal aortic aneurysm
- Diagnosing an abdominal aortic aneurysm
- Treating an abdominal aortic aneurysm
- Preventing an abdominal aortic aneurysm


In [22]:
all_chunks[:5]

[{'chunk_id': 'abdominal-aortic-aneurysm-1',
  'parent_id': 'abdominal-aortic-aneurysm',
  'category': 'symptom',
  'section': 'Abdominal aortic aneurysm',
  'heading': '# Abdominal aortic aneurysm',
  'url': 'https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/abdominal-aortic-aneurysm/',
  'content': '# Abdominal aortic aneurysm\n\n- About abdominal aortic aneurysms\n- Symptoms of an abdominal aortic aneurysm\n- Causes of an abdominal aortic aneurysm\n- Diagnosing an abdominal aortic aneurysm\n- Treating an abdominal aortic aneurysm\n- Preventing an abdominal aortic aneurysm'},
 {'chunk_id': 'abdominal-aortic-aneurysm-2',
  'parent_id': 'abdominal-aortic-aneurysm',
  'category': 'symptom',
  'section': 'Abdominal aortic aneurysm',
  'heading': 'About abdominal aortic aneurysms',
  'url': 'https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/abdominal-aortic-aneurysm/',
  'content': '## About abdominal aortic aneurysms\n\nAn abdominal aortic aneurysm (AAA) is a swelling (